# File-Backed Performance Profile

This notebook profiles a fixed synthetic HDF5 spectrogram so file-backed
operations can be compared directly against the default eager
`Frame(waterfall=...)` path.

The timings are intentionally simple wall-clock measurements. They are
useful for internal reporting and regression checks, but they are not a
substitute for a dedicated benchmark suite. Run from a fresh kernel and
keep the demo dimensions fixed when comparing branches or machines.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython import get_ipython
from IPython.display import display
import matplotlib
_ipython = get_ipython()
if _ipython is not None:
    _ipython.run_line_magic("matplotlib", "inline")
    matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import setigen as stg

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

np.set_printoptions(precision=4, suppress=True)
print("matplotlib backend:", matplotlib.get_backend())
print("setigen from:", stg.__file__)

In [ ]:
import gc
import shutil
import time
import tracemalloc

import pandas as pd
import psutil

PROCESS = psutil.Process()

PROFILE_TCHANS = 24
PROFILE_FCHANS = 2**16
PROFILE_DF = 1 * u.Hz
PROFILE_DT = 1 * u.s
PROFILE_FCH1 = (6e9 + PROFILE_FCHANS - 1) * u.Hz
PROFILE_CHUNK_BYTES = 512 * 1024
PROFILE_CONTEXT_WIDTH = 2048
PROFILE_GUARD_WIDTH = 64
PROFILE_BOUND_HALF_WIDTH = 256
REBUILD_SOURCE = True

source_path = OUT / "profile_source.h5"

def rss_mib():
    return PROCESS.memory_info().rss / 1024**2

def profile_case(label, func, records, **metadata):
    gc.collect()
    rss_before = rss_mib()
    tracemalloc.start()
    t0 = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rss_after = rss_mib()

    record = {
        "case": label,
        "elapsed_s": elapsed,
        "rss_delta_mib": rss_after - rss_before,
        "python_peak_mib": peak / 1024**2,
    }
    record.update(metadata)
    records.append(record)
    return result

def show_table(records, sort_by=None):
    table = pd.DataFrame(records)
    if sort_by is not None:
        table = table.sort_values(sort_by)
    display(table.round({
        "elapsed_s": 4,
        "rss_delta_mib": 2,
        "python_peak_mib": 2,
    }))
    return table

def describe_shape(tchans, fchans, dtype=np.float64):
    bytes_ = tchans * fchans * np.dtype(dtype).itemsize
    return bytes_ / 1024**2

print("profile shape:", (PROFILE_TCHANS, PROFILE_FCHANS))
print("float64 materialized array, MiB:", describe_shape(PROFILE_TCHANS, PROFILE_FCHANS))
print("chunk budget, MiB:", PROFILE_CHUNK_BYTES / 1024**2)

In [ ]:
if REBUILD_SOURCE or not source_path.exists():
    source = stg.Frame(
        tchans=PROFILE_TCHANS,
        fchans=PROFILE_FCHANS,
        df=PROFILE_DF,
        dt=PROFILE_DT,
        fch1=PROFILE_FCH1,
        ascending=False,
        seed=61,
        source_name="Performance profile source",
    )
    source.add_noise(10, noise_type="chi2")
    source.save_hdf5(source_path)

print("source:", source_path)
print("source size, MiB:", source_path.stat().st_size / 1024**2)

In [ ]:
read_records = []
f0 = PROFILE_FCHANS // 2 - PROFILE_BOUND_HALF_WIDTH
f1 = PROFILE_FCHANS // 2 + PROFILE_BOUND_HALF_WIDTH

def eager_full_load():
    return stg.Frame(waterfall=source_path)

def file_backed_bounded_read():
    with stg.Frame.open(source_path, mode="r", max_chunk_bytes=PROFILE_CHUNK_BYTES) as backed:
        return backed.read_frame(
            f_index_range=(f0, f1),
            t_index_range=(0, backed.tchans),
        )

def file_backed_full_read():
    with stg.Frame.open(source_path, mode="r", max_chunk_bytes=PROFILE_CHUNK_BYTES) as backed:
        return backed.read_frame()

eager = profile_case(
    "eager full load: Frame(waterfall=...)",
    eager_full_load,
    read_records,
    operation="read",
    fchans_read=PROFILE_FCHANS,
    tchans_read=PROFILE_TCHANS,
)
bounded = profile_case(
    "file-backed bounded read_frame",
    file_backed_bounded_read,
    read_records,
    operation="read",
    fchans_read=f1 - f0,
    tchans_read=PROFILE_TCHANS,
)
full_backed = profile_case(
    "file-backed full read_frame",
    file_backed_full_read,
    read_records,
    operation="read",
    fchans_read=PROFILE_FCHANS,
    tchans_read=PROFILE_TCHANS,
)

print("bounded shape:", bounded.shape)
print("full file-backed read matches eager:", np.allclose(full_backed.data, eager.data))
read_table = show_table(read_records)

In [ ]:
write_records = []

def eager_rewrite():
    frame = stg.Frame(waterfall=source_path)
    output = OUT / "profile_eager_rewrite.h5"
    frame.save_hdf5(output)
    return output

def file_backed_copy():
    output = OUT / "profile_file_backed_copy.h5"
    with stg.Frame.open_copy(source_path, output, overwrite=True, max_chunk_bytes=PROFILE_CHUNK_BYTES):
        pass
    return output

eager_out = profile_case(
    "eager load + save_hdf5",
    eager_rewrite,
    write_records,
    operation="write/copy",
    materializes_full_frame=True,
)
copy_out = profile_case(
    "file-backed open_copy",
    file_backed_copy,
    write_records,
    operation="write/copy",
    materializes_full_frame=False,
)

print("eager rewrite size, MiB:", eager_out.stat().st_size / 1024**2)
print("file-backed copy size, MiB:", copy_out.stat().st_size / 1024**2)
write_table = show_table(write_records)

In [ ]:
noise_records = []
noise_config = stg.NoiseEstimationConfig(
    method="sigma_clip",
    context_width=PROFILE_CONTEXT_WIDTH,
    guard_width=PROFILE_GUARD_WIDTH,
)

def local_noise_kwargs(frame):
    return {
        "path": stg.constant_path(
            f_start=frame.get_frequency(frame.fchans // 2),
            drift_rate=0.25 * frame.unit_drift_rate,
        ),
        "f_profile": stg.gaussian_f_profile(width=8 * frame.df),
        "auto_bounding": True,
        "truncate_below": 1e-3,
        "config": noise_config,
    }

def eager_noise_estimation():
    frame = stg.Frame(waterfall=source_path)
    return frame.estimate_noise_stats(**local_noise_kwargs(frame))

def file_backed_noise_estimation():
    with stg.Frame.open(source_path, mode="r", max_chunk_bytes=PROFILE_CHUNK_BYTES) as backed:
        return backed.estimate_noise_stats(**local_noise_kwargs(backed))

eager_noise = profile_case(
    "eager full load + local noise stats",
    eager_noise_estimation,
    noise_records,
    operation="noise estimation",
    materializes_full_frame=True,
)
backed_noise = profile_case(
    "file-backed local noise stats",
    file_backed_noise_estimation,
    noise_records,
    operation="noise estimation",
    materializes_full_frame=False,
)

print("eager noise stats:", eager_noise)
print("file-backed noise stats:", backed_noise)
noise_table = show_table(noise_records)

In [ ]:
try:
    with stg.Frame.open_copy(
        source_path,
        OUT / "profile_noise_mutation_attempt.h5",
        overwrite=True,
        max_chunk_bytes=PROFILE_CHUNK_BYTES,
    ) as backed:
        backed.add_noise(10, noise_type="chi2")
except NotImplementedError as exc:
    print("file-backed add_noise is intentionally unsupported:")
    print(exc)

In [ ]:
signal_records = []

def injection_kwargs(frame, *, auto_bounding):
    return {
        "path": stg.constant_path(
            f_start=frame.get_frequency(frame.fchans // 2),
            drift_rate=0.25 * frame.unit_drift_rate,
        ),
        "t_profile": stg.constant_t_profile(level=25),
        "f_profile": stg.gaussian_f_profile(width=8 * frame.df),
        "bp_profile": stg.constant_bp_profile(level=1),
        "auto_bounding": auto_bounding,
        "truncate_below": 1e-3,
    }

def affected_fchans_from_signal(signal):
    nonzero = np.flatnonzero(np.any(signal != 0, axis=0))
    return int(nonzero.size)

def eager_injection(auto_bounding):
    frame = stg.Frame(waterfall=source_path)
    signal = frame.add_signal(**injection_kwargs(frame, auto_bounding=auto_bounding))
    output = OUT / f"profile_eager_injected_auto_{auto_bounding}.h5"
    frame.save_hdf5(output)
    return {
        "output": output,
        "affected_fchans": affected_fchans_from_signal(signal),
        "returned_signal_shape": signal.shape,
    }

def file_backed_injection(auto_bounding):
    output = OUT / f"profile_file_backed_injected_auto_{auto_bounding}.h5"
    with stg.Frame.open_copy(
        source_path,
        output,
        overwrite=True,
        max_chunk_bytes=PROFILE_CHUNK_BYTES,
    ) as backed:
        result = backed.add_signal(
            **injection_kwargs(backed, auto_bounding=auto_bounding),
            max_chunk_bytes=PROFILE_CHUNK_BYTES,
        )
    return {"output": output, "result": result}

for auto_bounding in (False, True):
    eager_result = profile_case(
        f"eager injection auto_bounding={auto_bounding}",
        lambda auto_bounding=auto_bounding: eager_injection(auto_bounding),
        signal_records,
        operation="signal injection",
        storage="eager",
        auto_bounding=auto_bounding,
    )
    signal_records[-1]["affected_fchans"] = eager_result["affected_fchans"]
    signal_records[-1]["returned_signal_shape"] = eager_result["returned_signal_shape"]

    backed_result = profile_case(
        f"file-backed injection auto_bounding={auto_bounding}",
        lambda auto_bounding=auto_bounding: file_backed_injection(auto_bounding),
        signal_records,
        operation="signal injection",
        storage="file-backed",
        auto_bounding=auto_bounding,
    )
    fb_result = backed_result["result"]
    signal_records[-1]["affected_fchans"] = fb_result.frequency_slice.stop - fb_result.frequency_slice.start
    signal_records[-1]["time_chunks"] = fb_result.time_chunks
    signal_records[-1]["max_chunk_shape"] = fb_result.max_chunk_shape

signal_table = show_table(signal_records)

In [ ]:
all_tables = {
    "read": read_table,
    "write": write_table,
    "noise": noise_table,
    "signal": signal_table,
}

for name, table in all_tables.items():
    path = OUT / f"profile_{name}_table.csv"
    table.to_csv(path, index=False)
    print("wrote", path)

summary = pd.concat(
    [table.assign(section=name) for name, table in all_tables.items()],
    ignore_index=True,
    sort=False,
)
display(summary[[
    "section",
    "case",
    "elapsed_s",
    "rss_delta_mib",
    "python_peak_mib",
    "affected_fchans",
    "fchans_read",
    "materializes_full_frame",
]].round({
    "elapsed_s": 4,
    "rss_delta_mib": 2,
    "python_peak_mib": 2,
}))

## Analysis

The main question is not whether file-backed access is universally
faster. It should be comparable to eager loading when the whole
observation must be materialized. The expected win is when the
scientific operation is local in frequency: narrowband reads,
local noise/SNR estimation, and bounded signal injection should
touch only the relevant channels.

In [ ]:
def one(table, case):
    match = table.loc[table["case"] == case]
    if len(match) != 1:
        raise ValueError(f"Expected one row for {case!r}, found {len(match)}")
    return match.iloc[0]

def speedup(reference, candidate):
    return float(reference["elapsed_s"] / candidate["elapsed_s"])

def memory_ratio(reference, candidate, column):
    ref = float(reference[column])
    cand = float(candidate[column])
    if cand == 0:
        return np.inf
    return ref / cand

read_eager = one(read_table, "eager full load: Frame(waterfall=...)")
read_bounded = one(read_table, "file-backed bounded read_frame")
read_full_backed = one(read_table, "file-backed full read_frame")

write_eager = one(write_table, "eager load + save_hdf5")
write_copy = one(write_table, "file-backed open_copy")

noise_eager = one(noise_table, "eager full load + local noise stats")
noise_backed = one(noise_table, "file-backed local noise stats")

sig_eager_unbounded = one(signal_table, "eager injection auto_bounding=False")
sig_backed_unbounded = one(signal_table, "file-backed injection auto_bounding=False")
sig_eager_bounded = one(signal_table, "eager injection auto_bounding=True")
sig_backed_bounded = one(signal_table, "file-backed injection auto_bounding=True")

bounded_read_fraction = read_bounded["fchans_read"] / PROFILE_FCHANS
bounded_signal_fraction = sig_backed_bounded["affected_fchans"] / PROFILE_FCHANS

analysis_rows = [
    {
        "comparison": "bounded file-backed read vs eager full load",
        "wall_time_speedup": speedup(read_eager, read_bounded),
        "python_peak_ratio": memory_ratio(read_eager, read_bounded, "python_peak_mib"),
        "rss_delta_ratio": memory_ratio(read_eager, read_bounded, "rss_delta_mib"),
        "channels_touched_fraction": bounded_read_fraction,
    },
    {
        "comparison": "full file-backed read vs eager full load",
        "wall_time_speedup": speedup(read_eager, read_full_backed),
        "python_peak_ratio": memory_ratio(read_eager, read_full_backed, "python_peak_mib"),
        "rss_delta_ratio": memory_ratio(read_eager, read_full_backed, "rss_delta_mib"),
        "channels_touched_fraction": 1.0,
    },
    {
        "comparison": "file-backed open_copy vs eager load + save",
        "wall_time_speedup": speedup(write_eager, write_copy),
        "python_peak_ratio": memory_ratio(write_eager, write_copy, "python_peak_mib"),
        "rss_delta_ratio": memory_ratio(write_eager, write_copy, "rss_delta_mib"),
        "channels_touched_fraction": 1.0,
    },
    {
        "comparison": "file-backed local noise stats vs eager local noise stats",
        "wall_time_speedup": speedup(noise_eager, noise_backed),
        "python_peak_ratio": memory_ratio(noise_eager, noise_backed, "python_peak_mib"),
        "rss_delta_ratio": memory_ratio(noise_eager, noise_backed, "rss_delta_mib"),
        "channels_touched_fraction": (PROFILE_CONTEXT_WIDTH * 2 + 32) / PROFILE_FCHANS,
    },
    {
        "comparison": "file-backed bounded injection vs eager bounded injection",
        "wall_time_speedup": speedup(sig_eager_bounded, sig_backed_bounded),
        "python_peak_ratio": memory_ratio(sig_eager_bounded, sig_backed_bounded, "python_peak_mib"),
        "rss_delta_ratio": memory_ratio(sig_eager_bounded, sig_backed_bounded, "rss_delta_mib"),
        "channels_touched_fraction": bounded_signal_fraction,
    },
    {
        "comparison": "file-backed unbounded injection vs eager unbounded injection",
        "wall_time_speedup": speedup(sig_eager_unbounded, sig_backed_unbounded),
        "python_peak_ratio": memory_ratio(sig_eager_unbounded, sig_backed_unbounded, "python_peak_mib"),
        "rss_delta_ratio": memory_ratio(sig_eager_unbounded, sig_backed_unbounded, "rss_delta_mib"),
        "channels_touched_fraction": 1.0,
    },
    {
        "comparison": "file-backed bounded injection vs file-backed unbounded injection",
        "wall_time_speedup": speedup(sig_backed_unbounded, sig_backed_bounded),
        "python_peak_ratio": memory_ratio(sig_backed_unbounded, sig_backed_bounded, "python_peak_mib"),
        "rss_delta_ratio": memory_ratio(sig_backed_unbounded, sig_backed_bounded, "rss_delta_mib"),
        "channels_touched_fraction": bounded_signal_fraction,
    },
]

analysis_table = pd.DataFrame(analysis_rows)
display(analysis_table.round({
    "wall_time_speedup": 2,
    "python_peak_ratio": 2,
    "rss_delta_ratio": 2,
    "channels_touched_fraction": 5,
}))

analysis_path = OUT / "profile_analysis_table.csv"
analysis_table.to_csv(analysis_path, index=False)
print("wrote", analysis_path)

In [ ]:
from IPython.display import Markdown

def fmt_speed(value):
    return f"{value:.1f}x"

def fmt_pct(value):
    return f"{100 * value:.3f}%"

read_speed = speedup(read_eager, read_bounded)
noise_speed = speedup(noise_eager, noise_backed)
copy_speed = speedup(write_eager, write_copy)
bounded_signal_speed = speedup(sig_eager_bounded, sig_backed_bounded)
unbounded_signal_speed = speedup(sig_eager_unbounded, sig_backed_unbounded)
file_backed_bound_speed = speedup(sig_backed_unbounded, sig_backed_bounded)

interpretation = f'''
### Interpretation

- Bounded file-backed reads touched {int(read_bounded["fchans_read"]):,} of
  {PROFILE_FCHANS:,} channels ({fmt_pct(bounded_read_fraction)}) and were
  {fmt_speed(read_speed)} faster than eager full loading in this run.

- Full file-backed reads were intentionally similar to eager full loading
  ({fmt_speed(speedup(read_eager, read_full_backed))} relative speed).
  File backing is not magic when the operation truly needs every channel;
  the advantage appears when the read/write region is narrow.

- `Frame.open_copy(...)` avoided materializing the full observation and was
  {fmt_speed(copy_speed)} faster than eager load plus `save_hdf5(...)`.
  This is the right pattern for safe copy-backed mutation before injection.

- File-backed local noise estimation produced the same noise statistics as
  eager local noise estimation while avoiding the full-frame load. In this
  run it was {fmt_speed(noise_speed)} faster. This is the SNR-estimation
  path we care about scientifically because local context is needed around
  the signal track.

- Full-frame file-backed synthetic noise mutation is intentionally not
  implemented. Adding noise to an entire observation is a global synthetic
  operation and would require writing the whole file; for that workflow, use
  an eager synthetic frame or a future dedicated full-file generation path.

- With `auto_bounding=True`, file-backed injection touched
  {int(sig_backed_bounded["affected_fchans"]):,} of {PROFILE_FCHANS:,}
  channels ({fmt_pct(bounded_signal_fraction)}) and was
  {fmt_speed(bounded_signal_speed)} faster than eager bounded injection.
  It was also {fmt_speed(file_backed_bound_speed)} faster than file-backed
  unbounded injection.

- With `auto_bounding=False`, file-backed injection still avoids returning
  a full signal array, but it must read and write all frequency channels in
  chunks. That mode is mostly a correctness fallback; bounded injection is
  the performance-critical path for narrowband signals.

- RSS deltas and wall times are machine- and cache-dependent. The most
  stable reporting quantities are the channel fraction touched, whether the
  operation materializes the full frame, and Python peak allocation from
  `tracemalloc`.
'''

display(Markdown(interpretation))